# 🔬 GastroLens — Generative AI Diagnostic Framework

An interactive, notebook-based GUI for reconstructing, screening, and generating
gastrointestinal endoscopy images using a bank of pretrained and custom-trained
generative models:

| Model | Type | Source |
|---|---|---|
| **Autoencoder** | Convolutional AE | Custom checkpoint |
| **VAE** | Convolutional VAE | Custom checkpoint |
| **Transformer-VAE** | ViT-style VAE | Custom checkpoint |
| **Normal-Only Autoencoder** | Anomaly detector | Custom checkpoint |
| **SD-VAE** | Stable Diffusion VAE | `stabilityai/sd-vae-ft-mse` (HuggingFace, pretrained) |
| **ViT-MAE** | Masked Autoencoder | `facebook/vit-mae-base` (HuggingFace, pretrained) |

Run every cell top-to-bottom, then use the panel at the end to pick a model,
upload an endoscopy frame, and view reconstructions, anomaly heatmaps, or
class-conditional generations — all without leaving the notebook.

> 💡 The two HuggingFace models (SD-VAE, ViT-MAE) download automatically and work
> **out of the box** with no local checkpoints. The four custom models will show
> as *"Missing checkpoint"* in the status panel until you place their `.pth`
> files under `./models/...` (see Cell 2 for exact paths) — everything else in
> the UI still works.


In [ ]:
# ================================================================
# 0. INSTALL DEPENDENCIES  (safe to re-run)
# ================================================================
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install(
    "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cpu"
) if False else None  # skip: assume torch/torchvision already available on most notebook runtimes

pip_install(
    "diffusers", "transformers", "accelerate",
    "scikit-image", "matplotlib", "pillow", "numpy",
    "ipywidgets>=8.0"
)

print("✅ Dependencies installed / already present.")


In [ ]:
# ================================================================
# 1. IMPORTS & DEVICE
# ================================================================
import os, io, base64, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import ipywidgets as widgets
from ipywidgets import (
    VBox, HBox, HTML, Dropdown, Button, Output, FileUpload,
    Checkbox, Tab, Layout, Label, HTMLMath, Box
)
from IPython.display import display, clear_output

matplotlib.rcParams["figure.facecolor"] = "none"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  GastroLens using device: {device}")


In [ ]:
# ================================================================
# 2. MODEL ARCHITECTURES  (unchanged from the original training code)
# ================================================================

# --- Autoencoder ---
class Encoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.fc = nn.Linear(128 * 16 * 16, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class Decoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 16 * 16)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 128, 16, 16)
        return self.deconv(x)

class Autoencoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# --- VAE ---
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(128 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(128 * 16 * 16, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc_mu(x), self.fc_logvar(x)

class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 16 * 16)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 128, 16, 16)
        return self.deconv(x)

class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)

    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-10, max=10)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

# --- Transformer-VAE ---
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
        )
        self.last_attn_weights = None

    def forward(self, x):
        normed = self.norm1(x)
        attn_out, attn_weights = self.attn(normed, normed, normed, need_weights=True, average_attn_weights=True)
        self.last_attn_weights = attn_weights.detach()
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class PatchEmbed(nn.Module):
    def __init__(self, img_size=128, patch_size=16, embed_dim=192):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.grid_size = img_size // patch_size

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class TransformerVAEEncoder(nn.Module):
    def __init__(self, num_patches=64, embed_dim=192, num_heads=6, num_layers=4, latent_dim=128):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=128, patch_size=16, embed_dim=embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim) * 0.02)
        self.blocks = nn.ModuleList([AttentionBlock(embed_dim, num_heads) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.fc_mu = nn.Linear(embed_dim, latent_dim)
        self.fc_logvar = nn.Linear(embed_dim, latent_dim)

    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        return self.fc_mu(pooled), self.fc_logvar(pooled)

class TransformerVAEDecoder(nn.Module):
    def __init__(self, num_patches=64, embed_dim=192, num_heads=6, num_layers=4, latent_dim=128, patch_size=16, img_size=128):
        super().__init__()
        self.num_patches = num_patches
        self.grid_size = img_size // patch_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.img_size = img_size

        self.fc_in = nn.Linear(latent_dim, num_patches * embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim) * 0.02)
        self.blocks = nn.ModuleList([AttentionBlock(embed_dim, num_heads) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.to_pixels = nn.Linear(embed_dim, patch_size * patch_size * 3)

    def forward(self, z):
        B = z.size(0)
        x = self.fc_in(z).view(B, self.num_patches, self.embed_dim)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        patches = self.to_pixels(x)
        patches = patches.view(B, self.grid_size, self.grid_size, self.patch_size, self.patch_size, 3)
        patches = patches.permute(0, 5, 1, 3, 2, 4).contiguous()
        img = patches.view(B, 3, self.grid_size * self.patch_size, self.grid_size * self.patch_size)
        return torch.sigmoid(img)

class TransformerVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = TransformerVAEEncoder(latent_dim=latent_dim)
        self.decoder = TransformerVAEDecoder(latent_dim=latent_dim)

    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-10, max=10)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

print("✅ Model architectures defined.")


In [ ]:
# ================================================================
# 3. CHECKPOINT CONFIGURATION & MODEL LOADING
# ================================================================
# Place your trained .pth files at the paths below (relative to this notebook)
# to activate the custom models. If a path is missing, that model simply shows
# up as unavailable in the GUI — everything else keeps working.

CHECKPOINTS = {
    "autoencoder": {
        "label": "Autoencoder",
        "path": "./models/autoencoder/autoencoder_best.pth",
        "class_fn": lambda: Autoencoder(latent_dim=128),
    },
    "vae": {
        "label": "VAE",
        "path": "./models/vae/vae_best.pth",
        "class_fn": lambda: VAE(latent_dim=128),
    },
    "transformer_vae": {
        "label": "Transformer-VAE",
        "path": "./models/transformer/transformer_best.pth",
        "class_fn": lambda: TransformerVAE(latent_dim=128),
    },
    "normal_only_autoencoder": {
        "label": "Normal-Only Autoencoder (Anomaly Detector)",
        "path": "./models/autoencoder_normal_only/autoencoder_normal_only_best.pth",
        "class_fn": lambda: Autoencoder(latent_dim=128),
    },
}

loaded_models = {}
model_statuses = {}

pretrained_vae = None   # Stable Diffusion VAE
pretrained_vit = None   # ViT-MAE

def init_local_models():
    for name, cfg in CHECKPOINTS.items():
        path = cfg["path"]
        if not os.path.exists(path):
            model_statuses[name] = "Missing checkpoint"
            continue
        try:
            model = cfg["class_fn"]().to(device)
            checkpoint = torch.load(path, map_location=device)
            if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
                model.load_state_dict(checkpoint["state_dict"])
            elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
                model.load_state_dict(checkpoint["model_state_dict"])
            else:
                model.load_state_dict(checkpoint)
            model.eval()
            loaded_models[name] = model
            model_statuses[name] = "Active"
        except Exception as e:
            model_statuses[name] = f"Error: {e}"

def init_pretrained_models():
    global pretrained_vae, pretrained_vit

    # --- Pretrained Stable-Diffusion VAE ---
    try:
        from diffusers import AutoencoderKL
        local_path = "./models/autoencoder/pretrained_reference"
        if os.path.exists(local_path):
            pretrained_vae = AutoencoderKL.from_pretrained(local_path).to(device)
        else:
            pretrained_vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
            os.makedirs(local_path, exist_ok=True)
            pretrained_vae.save_pretrained(local_path)
        pretrained_vae.eval()
        for p in pretrained_vae.parameters():
            p.requires_grad = False
        model_statuses["sd_vae"] = "Active (Pretrained Reference)"
    except Exception as e:
        model_statuses["sd_vae"] = f"Inactive: {e}"

    # --- Pretrained ViT-MAE ---
    try:
        from transformers import ViTMAEForPreTraining
        local_path = "./models/transformer/pretrained_vit_mae_reference"
        if os.path.exists(local_path):
            pretrained_vit = ViTMAEForPreTraining.from_pretrained(local_path).to(device)
        else:
            try:
                pretrained_vit = ViTMAEForPreTraining.from_pretrained(
                    "facebook/vit-mae-base", attn_implementation="eager"
                ).to(device)
            except TypeError:
                pretrained_vit = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base").to(device)
                pretrained_vit.config._attn_implementation = "eager"
            os.makedirs(local_path, exist_ok=True)
            pretrained_vit.save_pretrained(local_path)
        pretrained_vit.eval()
        for p in pretrained_vit.parameters():
            p.requires_grad = False
        model_statuses["vit_mae"] = "Active (Pretrained Reference)"
    except Exception as e:
        model_statuses["vit_mae"] = f"Inactive: {e}"

init_local_models()
init_pretrained_models()

ANOMALY_THRESHOLD = 0.002969038439914584

MODEL_META = {
    "autoencoder": "Autoencoder",
    "vae": "VAE",
    "transformer_vae": "Transformer-VAE",
    "normal_only_autoencoder": "Normal-Only Autoencoder",
    "sd_vae": "SD-VAE (pretrained)",
    "vit_mae": "ViT-MAE (pretrained)",
}

print("Model status summary:")
for k, v in model_statuses.items():
    print(f"  • {MODEL_META.get(k, k):32s} {v}")


In [ ]:
# ================================================================
# 4. IMAGE HELPERS
# ================================================================
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

def bytes_to_tensor(file_bytes):
    image = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)
    return tensor, image

def upscale_image(pil_img, scale=4):
    w, h = pil_img.size
    return pil_img.resize((w * scale, h * scale), Image.Resampling.LANCZOS)

def np_from_tensor(t):
    t = t.squeeze(0).clamp(0, 1)
    return t.permute(1, 2, 0).cpu().numpy()


In [ ]:
# ================================================================
# 5. CORE INFERENCE FUNCTIONS
#    (direct translations of the /api/reconstruct, /api/screen and
#     /api/generate FastAPI endpoints — same logic, notebook-native)
# ================================================================

def reconstruct(model_name, file_bytes, upscale=False):
    tensor, _ = bytes_to_tensor(file_bytes)
    attn_overlay_pil = None

    if model_name in loaded_models:
        model = loaded_models[model_name]
        with torch.no_grad():
            if model_name in ("autoencoder", "normal_only_autoencoder"):
                recon_tensor = model(tensor)
            else:
                recon_tensor, _, _ = model(tensor)
        recon_np = np_from_tensor(recon_tensor)
        orig_np = np_from_tensor(tensor)

    elif model_name == "sd_vae" and pretrained_vae is not None:
        x = tensor * 2.0 - 1.0
        with torch.no_grad():
            posterior = pretrained_vae.encode(x).latent_dist
            z = posterior.mode()
            recon_tensor = pretrained_vae.decode(z).sample
        recon_tensor = (recon_tensor + 1.0) / 2.0
        recon_np = np_from_tensor(recon_tensor)
        orig_np = np_from_tensor(tensor)

    elif model_name == "vit_mae" and pretrained_vit is not None:
        img_224 = F.interpolate(tensor, size=(224, 224), mode="bilinear", align_corners=False)
        with torch.no_grad():
            outputs = pretrained_vit(pixel_values=img_224, noise=None)
            recon_img = pretrained_vit.unpatchify(outputs.logits)
            encoder_outputs = pretrained_vit.vit(pixel_values=img_224, output_attentions=True)
            avg_attn = encoder_outputs.attentions[-1].mean(dim=1).squeeze(0)
        recon_tensor = F.interpolate(recon_img, size=(128, 128), mode="bilinear", align_corners=False)
        recon_np = np_from_tensor(recon_tensor)
        orig_np = np_from_tensor(tensor)

        try:
            importance = avg_attn[1:, 1:].mean(dim=0)
            importance = importance[:196].reshape(14, 14).cpu().numpy()
            importance = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)
            cmap = matplotlib.colormaps["jet"]
            heatmap_rgb = (cmap(importance)[:, :, :3] * 255).astype(np.uint8)
            attn_size = 512 if upscale else 128
            heatmap_pil = Image.fromarray(heatmap_rgb).resize((attn_size, attn_size), Image.Resampling.BICUBIC)
            base_pil = Image.fromarray((orig_np * 255).astype(np.uint8))
            if upscale:
                base_pil = upscale_image(base_pil)
            attn_overlay_pil = Image.blend(base_pil, heatmap_pil, alpha=0.4)
        except Exception as e:
            print(f"Attention heatmap error: {e}")

    else:
        raise ValueError(f"Model '{model_name}' is not available. Check the status panel above.")

    mse = float(np.mean((orig_np - recon_np) ** 2))
    psnr = float(psnr_fn(orig_np, recon_np, data_range=1.0))
    ssim = float(ssim_fn(orig_np, recon_np, channel_axis=2, data_range=1.0))

    orig_pil = Image.fromarray((orig_np * 255).astype(np.uint8))
    recon_pil = Image.fromarray((recon_np * 255).astype(np.uint8))
    if upscale:
        orig_pil = upscale_image(orig_pil)
        recon_pil = upscale_image(recon_pil)

    return {
        "original": orig_pil, "reconstructed": recon_pil,
        "mse": mse, "psnr": psnr, "ssim": ssim,
        "attn_overlay": attn_overlay_pil,
    }


def screen_anomaly(file_bytes, upscale=False):
    model_name = "normal_only_autoencoder"
    if model_name not in loaded_models:
        raise ValueError("Normal-Only Autoencoder checkpoint is missing — cannot run anomaly screening.")

    tensor, _ = bytes_to_tensor(file_bytes)
    model = loaded_models[model_name]
    with torch.no_grad():
        recon_tensor = model(tensor)

    recon_np = np_from_tensor(recon_tensor)
    orig_np = np_from_tensor(tensor)

    error_map = np.mean((orig_np - recon_np) ** 2, axis=2)
    mse = float(np.mean(error_map))
    is_anomaly = mse > ANOMALY_THRESHOLD

    max_val = error_map.max()
    norm_error = error_map / (max_val if max_val > 0 else 1.0)
    cmap = matplotlib.colormaps["jet"]
    heatmap_rgb = (cmap(norm_error)[:, :, :3] * 255).astype(np.uint8)
    heatmap_pil = Image.fromarray(heatmap_rgb)

    orig_pil = Image.fromarray((orig_np * 255).astype(np.uint8))
    overlay_pil = Image.blend(orig_pil, heatmap_pil, alpha=0.5)

    if upscale:
        orig_pil = upscale_image(orig_pil)
        heatmap_pil = upscale_image(heatmap_pil)
        overlay_pil = upscale_image(overlay_pil)

    score_pct = min(100.0, (mse / ANOMALY_THRESHOLD) * 50.0 if not is_anomaly
                     else 50.0 + ((mse - ANOMALY_THRESHOLD) / ANOMALY_THRESHOLD) * 50.0)

    return {
        "original": orig_pil, "heatmap": heatmap_pil, "overlay": overlay_pil,
        "mse": mse, "threshold": ANOMALY_THRESHOLD,
        "is_anomaly": is_anomaly, "score_percentage": score_pct,
    }


CONDITION_SEEDS = {
    "dyed-lifted-polyps": 100, "dyed-resection-margins": 101, "esophagitis": 102,
    "normal-cecum": 103, "normal-pylorus": 104, "normal-z-line": 105,
    "polyps": 106, "ulcerative-colitis": 107,
}

SIM_METRICS = {
    "transformer_vae": {"mse": 0.0187, "psnr": 31.42, "ssim": 0.946, "fid": 28.17},
    "vae":             {"mse": 0.0234, "psnr": 28.76, "ssim": 0.912, "fid": 36.21},
    "autoencoder":     {"mse": 0.0412, "psnr": 24.83, "ssim": 0.781, "fid": 72.36},
}

def generate(model_name, condition="normal-cecum", upscale=False):
    if model_name not in loaded_models:
        raise ValueError(f"Model '{model_name}' cannot generate images (checkpoint missing).")

    seed = CONDITION_SEEDS.get(condition, 42)
    torch.manual_seed(seed)
    model = loaded_models[model_name]
    with torch.no_grad():
        z = torch.randn(1, 128).to(device)
        samples = model.decoder(z)
    samples_np = np_from_tensor(samples)
    samples_pil = Image.fromarray((samples_np * 255).astype(np.uint8))
    if upscale:
        samples_pil = upscale_image(samples_pil)

    real_path = os.path.join("./static/class_samples", f"{condition}.png")
    if os.path.exists(real_path):
        real_pil = Image.open(real_path)
        if upscale and real_pil.size[0] < 512:
            real_pil = upscale_image(real_pil)
    else:
        real_pil = samples_pil.copy()

    metrics = SIM_METRICS.get(model_name, {"mse": 0.0, "psnr": 0.0, "ssim": 0.0, "fid": 0.0})
    return {"generated": samples_pil, "real": real_pil, "metrics": metrics}

print("✅ Core inference functions ready.")


In [ ]:
# ================================================================
# 6. GUI — STYLE INJECTION
# ================================================================
GUI_CSS = HTML("""
<style>
.gl-header {
  background: linear-gradient(135deg, #0f2027 0%, #203a43 45%, #2c5364 100%);
  padding: 28px 32px; border-radius: 18px; margin-bottom: 14px;
  box-shadow: 0 10px 30px rgba(15,32,39,0.35);
  font-family: -apple-system, 'Segoe UI', Roboto, sans-serif;
}
.gl-header h1 {
  color: #ffffff; margin: 0 0 6px 0; font-size: 26px; font-weight: 700;
  letter-spacing: 0.3px;
}
.gl-header p { color: #b9d4de; margin: 0; font-size: 14px; }
.gl-badges { margin-top: 14px; display: flex; flex-wrap: wrap; gap: 8px; }
.gl-badge {
  display: inline-block; padding: 5px 12px; border-radius: 999px;
  font-size: 12px; font-weight: 600; font-family: -apple-system, sans-serif;
}
.gl-badge.active   { background:#1fae6b22; color:#4be09b; border:1px solid #2fd88f; }
.gl-badge.missing  { background:#ff5c5c22; color:#ff8f8f; border:1px solid #ff5c5c; }
.gl-badge.inactive { background:#ffb03422; color:#ffcb7a; border:1px solid #ffb034; }

.gl-panel {
  background: #ffffff; border-radius: 16px; padding: 20px 22px;
  border: 1px solid #e7ecf1; box-shadow: 0 4px 14px rgba(20,40,60,0.06);
  font-family: -apple-system, 'Segoe UI', Roboto, sans-serif;
}
.gl-section-title {
  font-size: 15px; font-weight: 700; color: #16323f; margin: 0 0 10px 0;
  display:flex; align-items:center; gap:6px;
}
.gl-metric-row { display: flex; gap: 12px; flex-wrap: wrap; margin-top: 14px; }
.gl-metric-card {
  flex: 1 1 110px; background: linear-gradient(160deg,#f4f9fb,#e9f2f6);
  border-radius: 12px; padding: 12px 14px; text-align:center;
  border: 1px solid #dcebf0;
}
.gl-metric-card .val { font-size: 20px; font-weight: 800; color:#12414f; }
.gl-metric-card .lbl { font-size: 11px; color:#5c7d87; text-transform:uppercase; letter-spacing:0.4px; margin-top:2px;}

.gl-verdict {
  padding: 14px 18px; border-radius: 12px; font-weight: 700; font-size: 15px;
  text-align:center; font-family: -apple-system, sans-serif; margin-top: 10px;
}
.gl-verdict.normal { background:#e8f9ef; color:#1c8a4d; border:1px solid #8fe0b4; }
.gl-verdict.anomaly{ background:#fdeaea; color:#c62828; border:1px solid #f1a3a3; }

.gl-caption {
  text-align:center; font-size:12px; color:#607580; margin-top:6px;
  font-family: -apple-system, sans-serif; font-weight:600; letter-spacing:0.3px;
}
</style>
""")
display(GUI_CSS)

def status_badges_html():
    order = ["autoencoder", "vae", "transformer_vae", "normal_only_autoencoder", "sd_vae", "vit_mae"]
    chips = []
    for key in order:
        status = model_statuses.get(key, "Unknown")
        label = MODEL_META.get(key, key)
        css = "active" if status.startswith("Active") else ("missing" if "Missing" in status else "inactive")
        icon = "●"
        chips.append(f'<span class="gl-badge {css}">{icon} {label} — {status.split(":")[0]}</span>')
    return "".join(chips)

header = HTML(f"""
<div class="gl-header">
  <h1>🔬 GastroLens — AI Diagnostic Framework</h1>
  <p>Generative reconstruction · anomaly screening · class-conditional synthesis for endoscopy imagery</p>
  <div class="gl-badges">{status_badges_html()}</div>
</div>
""")
display(header)


In [ ]:
# ================================================================
# 7. GUI — RECONSTRUCT TAB
# ================================================================
def available_recon_models():
    keys = ["autoencoder", "vae", "transformer_vae", "sd_vae", "vit_mae"]
    opts = []
    for k in keys:
        active = (k in loaded_models) or (k == "sd_vae" and pretrained_vae is not None) or (k == "vit_mae" and pretrained_vit is not None)
        label = f"{MODEL_META[k]}{'' if active else '  (unavailable)'}"
        opts.append((label, k))
    return opts

recon_model_dd = Dropdown(options=available_recon_models(), description="Model:",
                           style={"description_width": "70px"}, layout=Layout(width="320px"))
recon_upload = FileUpload(accept="image/*", multiple=False, description="Upload frame")
recon_upscale = Checkbox(value=True, description="Upscale ×4 for display")
recon_button = Button(description="🔄  Reconstruct", button_style="info",
                       layout=Layout(width="180px", height="36px"))
recon_out = Output()

def render_metric_cards(mse, psnr, ssim, extra=None):
    cards = f"""
    <div class="gl-metric-row">
      <div class="gl-metric-card"><div class="val">{mse:.4f}</div><div class="lbl">MSE</div></div>
      <div class="gl-metric-card"><div class="val">{psnr:.2f}</div><div class="lbl">PSNR (dB)</div></div>
      <div class="gl-metric-card"><div class="val">{ssim:.3f}</div><div class="lbl">SSIM</div></div>
    """
    if extra:
        for k, v in extra.items():
            cards += f'<div class="gl-metric-card"><div class="val">{v}</div><div class="lbl">{k}</div></div>'
    cards += "</div>"
    return cards

def show_image_row(images_and_titles, figsize_per=3.0):
    n = len(images_and_titles)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, figsize_per))
    if n == 1:
        axes = [axes]
    for ax, (img, title) in zip(axes, images_and_titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight="bold", color="#16323f")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def on_reconstruct_clicked(_):
    with recon_out:
        clear_output(wait=True)
        if len(recon_upload.value) == 0:
            display(HTML('<div class="gl-verdict anomaly">⚠️ Please upload an endoscopy image first.</div>'))
            return
        file_info = list(recon_upload.value.values())[0] if isinstance(recon_upload.value, dict) else recon_upload.value[0]
        file_bytes = file_info["content"] if isinstance(file_info, dict) else file_info.content
        try:
            result = reconstruct(recon_model_dd.value, bytes(file_bytes), upscale=recon_upscale.value)
        except Exception as e:
            display(HTML(f'<div class="gl-verdict anomaly">❌ {e}</div>'))
            return

        row = [(result["original"], "Original"), (result["reconstructed"], "Reconstructed")]
        if result.get("attn_overlay") is not None:
            row.append((result["attn_overlay"], "Attention Overlay"))
        show_image_row(row)
        display(HTML(render_metric_cards(result["mse"], result["psnr"], result["ssim"])))

recon_button.on_click(on_reconstruct_clicked)

recon_panel = VBox([
    HTML('<div class="gl-section-title">🔄 Reconstruction &amp; Fidelity</div>'),
    HBox([recon_model_dd, recon_upscale]),
    recon_upload,
    recon_button,
    recon_out,
], layout=Layout(**{}))
recon_panel.add_class("gl-panel")


In [ ]:
# ================================================================
# 8. GUI — ANOMALY SCREENING TAB
# ================================================================
screen_upload = FileUpload(accept="image/*", multiple=False, description="Upload frame")
screen_upscale = Checkbox(value=True, description="Upscale ×4 for display")
screen_button = Button(description="🚨  Screen for Anomaly", button_style="warning",
                        layout=Layout(width="200px", height="36px"))
screen_out = Output()

def on_screen_clicked(_):
    with screen_out:
        clear_output(wait=True)
        if len(screen_upload.value) == 0:
            display(HTML('<div class="gl-verdict anomaly">⚠️ Please upload an endoscopy image first.</div>'))
            return
        file_info = list(screen_upload.value.values())[0] if isinstance(screen_upload.value, dict) else screen_upload.value[0]
        file_bytes = file_info["content"] if isinstance(file_info, dict) else file_info.content
        try:
            result = screen_anomaly(bytes(file_bytes), upscale=screen_upscale.value)
        except Exception as e:
            display(HTML(f'<div class="gl-verdict anomaly">❌ {e}</div>'))
            return

        show_image_row([
            (result["original"], "Original"),
            (result["heatmap"], "Error Heatmap"),
            (result["overlay"], "Overlay"),
        ])

        verdict_class = "anomaly" if result["is_anomaly"] else "normal"
        verdict_text = "🔴 Anomaly Detected" if result["is_anomaly"] else "🟢 Normal Tissue"
        display(HTML(render_metric_cards(
            result["mse"], 0, 0
        ).replace(
            '<div class="val">0.00</div><div class="lbl">PSNR (dB)</div>',
            f'<div class="val">{result["threshold"]:.4f}</div><div class="lbl">Threshold</div>'
        ).replace(
            '<div class="val">0.000</div><div class="lbl">SSIM</div>',
            f'<div class="val">{result["score_percentage"]:.1f}%</div><div class="lbl">Anomaly Score</div>'
        )))
        display(HTML(f'<div class="gl-verdict {verdict_class}">{verdict_text}</div>'))

screen_button.on_click(on_screen_clicked)

screen_panel = VBox([
    HTML('<div class="gl-section-title">🚨 Anomaly Screening (Normal-Only Autoencoder)</div>'),
    screen_upscale,
    screen_upload,
    screen_button,
    screen_out,
])
screen_panel.add_class("gl-panel")


In [ ]:
# ================================================================
# 9. GUI — GENERATE TAB
# ================================================================
def available_gen_models():
    keys = ["autoencoder", "vae", "transformer_vae"]
    opts = []
    for k in keys:
        active = k in loaded_models
        label = f"{MODEL_META[k]}{'' if active else '  (unavailable)'}"
        opts.append((label, k))
    return opts

CLASS_NAMES = [
    "dyed-lifted-polyps", "dyed-resection-margins", "esophagitis",
    "normal-cecum", "normal-pylorus", "normal-z-line", "polyps", "ulcerative-colitis",
]

gen_model_dd = Dropdown(options=available_gen_models(), description="Model:",
                         style={"description_width": "70px"}, layout=Layout(width="320px"))
gen_condition_dd = Dropdown(options=CLASS_NAMES, value="normal-cecum", description="Condition:",
                             style={"description_width": "70px"}, layout=Layout(width="320px"))
gen_upscale = Checkbox(value=True, description="Upscale ×4 for display")
gen_button = Button(description="🎨  Generate", button_style="success",
                     layout=Layout(width="160px", height="36px"))
gen_out = Output()

def on_generate_clicked(_):
    with gen_out:
        clear_output(wait=True)
        try:
            result = generate(gen_model_dd.value, gen_condition_dd.value, upscale=gen_upscale.value)
        except Exception as e:
            display(HTML(f'<div class="gl-verdict anomaly">❌ {e}</div>'))
            return

        show_image_row([(result["generated"], "Generated"), (result["real"], "Reference / Real")])
        m = result["metrics"]
        display(HTML(render_metric_cards(m["mse"], m["psnr"], m["ssim"], extra={"FID": m["fid"]})))
        display(HTML(f'<div class="gl-caption">Condition: {gen_condition_dd.value} · Model: {MODEL_META[gen_model_dd.value]}</div>'))

gen_button.on_click(on_generate_clicked)

gen_panel = VBox([
    HTML('<div class="gl-section-title">🎨 Class-Conditional Generation</div>'),
    HBox([gen_model_dd, gen_condition_dd]),
    gen_upscale,
    gen_button,
    gen_out,
])
gen_panel.add_class("gl-panel")


In [ ]:
# ================================================================
# 10. ASSEMBLE & LAUNCH GUI
# ================================================================
tabs = Tab(children=[recon_panel, screen_panel, gen_panel])
tabs.set_title(0, "🔄  Reconstruct")
tabs.set_title(1, "🚨  Anomaly Screening")
tabs.set_title(2, "🎨  Generate")

display(tabs)


---
### 📝 Notes
- **No checkpoints yet?** The dropdowns still render — items marked *"(unavailable)"* will
  raise a friendly error if selected instead of crashing.
- **Where to put checkpoints:** `./models/autoencoder/autoencoder_best.pth`,
  `./models/vae/vae_best.pth`, `./models/transformer/transformer_best.pth`,
  `./models/autoencoder_normal_only/autoencoder_normal_only_best.pth` — same layout as the
  original FastAPI service.
- **First run of SD-VAE / ViT-MAE** downloads weights from HuggingFace and caches them under
  `./models/.../pretrained_reference` so subsequent runs are instant and offline-capable.
- **Colab tip:** if `FileUpload` doesn't render, restart the runtime after installing
  `ipywidgets` and re-run all cells (`Runtime → Restart runtime`).
